# Traffic-safety cameras in a simulated New York: the numbers, without a server

Every number in this folder's README, recomputed from the JSON files in `output/`. Nothing here needs the
simulator or a GPU. The last section reads a recording through `FileSource` if you have one locally (`record.py`
writes it; recordings are not committed).

Frames and labels: contains information from NYC Open Data, NY State open data and © OpenStreetMap contributors,
ODbL. Rendered with Valdrada v0.5.2 (MIT) and `--pedestrians procedural`.

In [ ]:
import json
from pathlib import Path

OUT = Path("output")


def load(name):
    return json.loads((OUT / name).read_text())

## 1. Seconds per labelled frame (`budget.py`)

P pole cameras capture on the same simulation step; the steps in between only advance the world. A labelled frame
is one camera's RGB plus instance labels with 8 amodal masks.

In [ ]:
budget = load("budget.json")
print("server GPU:", budget["gpu"])
print("poles  s/capture  s/frame  20k frames  idle tick  users/frame")
for r in budget["results"]:
    hours = 20_000 * r["wall_s_per_frame"] / 3600
    print(
        f"{r['poses']:>5} {r['wall_s_per_capture']:>10.2f} {r['wall_s_per_frame']:>8.2f}"
        f" {hours:>9.1f} h {r['idle_tick_s']:>9.3f}s {r['labels_per_frame']:>12.1f}"
    )

## 2. The data checks (`check3.py`)

A pose passes the pole search when under 10% of the frame is vegetation, under 5% is closer than 5 m, and over 20%
is road. Then the conditions: watch the mean pixel value of night against day (auto exposure).

In [ ]:
site = load("site_checks.json")
poles = site["pole_search"]
print(f"pole search: {sum(p['clear'] for p in poles)} of {len(poles)} candidate poses clear")
print("picked first:", {k: site["picked"][0][k] for k in ("bearing", "dist", "height")})
for c in site["conditions"]:
    print(f"{c['condition']:16s} road users {c['road_users']:3d}   mean RGB {c['mean_rgb']:6.1f}")
second, clip = site["load_second"], site["clip"]
print(f"second junction loaded in {second['seconds']:.1f} s: {second['junction']}")
print(f"clip: {clip['frames']} frames at {clip['fps']:.0f} fps in {clip['wall_s']:.0f} s")

## 3. A COCO detector on simulated frames (`zeroshot.py`)

RF-DETR Small with its COCO weights, confidence 0.5. A ground-truth box counts as found when a detection of a
matching class overlaps it at IoU ≥ 0.5. Occlusion is measured only for the 8 largest objects per frame, so the
occlusion slices hold large objects only.

In [ ]:
def table(*runs):
    names = sorted(set().union(*(r["slices"] for r in runs)))
    for sl in names:
        cells = [r["slices"].get(sl) for r in runs]
        n = next(c["n"] for c in cells if c)
        recalls = "  ".join(f"{c['recall']:.3f}" if c else "  -  " for c in cells)
        print(f"{sl:24s} n={n:5d}  {recalls}")
    fps = "  ".join(f"{r['fp_per_frame']:.2f} " for r in runs)
    print(f"{'false positives / frame':24s}         {fps}")


zs_all = load("zeroshot_all.json")
print(zs_all["frames"], "frames")
table(zs_all)

## 4. A first fine-tune (`to_coco.py`, `finetune.py`)

Split by capture run, so no split shares a frame or a 10 fps neighbour of one. Both models scored on the same test
split. Train and test come from the same junction: a pipeline check, not a claim about synthetic data.

In [ ]:
for split, s in load("splits.json").items():
    print(f"{split:5s} {s['images']:4d} frames {s['boxes']:5d} boxes  from {s['dirs']}")
print(load("finetune.json"))
print("\n                          COCO   fine-tuned")
table(load("zeroshot_test.json"), load("finetuned_test.json"))

In [ ]:
# Which class did each detection overlap? (checks that fine-tuned class ids map to names)
for name in ("zeroshot_test.json", "finetuned_test.json"):
    conf = load(name)["confusion"]
    print(name, dict(sorted(conf.items(), key=lambda kv: -kv[1])))

## 5. Record here, run anywhere (`sources.py`)

`open_source` takes a recording folder, a video file, an RTSP URL or a webcam index; the loop below is the same
for all of them. Set `REC` to a folder written by `record.py`, or to any video file.

In [ ]:
from sources import open_source

REC = "out/rec_day"  # or "some_video.mp4", "rtsp://...", 0
if Path(REC).exists() or not str(REC).startswith("out/"):
    src = open_source(REC)
    print(src.camera)
    for f in src:
        if f.index % 25 == 0:
            print(f.index, f"{f.t:.2f} s", f.image.shape)

## 6. Anatomy of one label (ground truth sidecar, evaluator only)

Each object has a visible box (`bbox`), the box of its whole outline including occluded parts (`amodal_bbox`, for
the 8 largest objects), the occluded fraction, whether it touches the image border, and the simulator's actor id
(0 for the static city: buildings, poles).

In [ ]:
gt_dir = Path(REC) / "gt"
if gt_dir.exists():
    lab = json.loads(sorted(gt_dir.glob("*.json"))[0].read_text())
    road = ("car", "bus", "truck", "bicycle", "pedestrian")
    users = [o for o in lab["objects"] if o["class_name"] in road]
    print(len(lab["objects"]), "objects,", len(users), "road users; K =", lab["camera"]["K"])
    for o in sorted(users, key=lambda o: -o["area"])[:5]:
        keys = ("class_name", "actor_id", "bbox", "amodal_bbox", "occlusion", "truncated", "area")
        print({k: o[k] for k in keys})